III. Preguntas de negocio que el modelo debe responder 
Una vez implementada la solución, las preguntas de negocio se deben responder 
generando los QUERIES (SQL) respectivos para cada grupo de preguntas. 
1. Análisis de Rendimiento y Tendencias de Ventas 
• Tendencias Temporales: Analizar el crecimiento o decrecimiento de las 
ventas (sale_dollars) por día, mes o año. 
• Ranking de Puntos de Venta: Determinar las tiendas y condados con 
mayores volúmenes de venta. 
• Comportamiento de Categorías: Identificar las categorías de productos 
(category_name) más vendidas y las de mayor crecimiento. 
2. B. Análisis de Rentabilidad (Gross Margin) 
• Cálculo de Margen: Utilizar el costo (state_bottle_cost) y el precio de venta 
(state_bottle_retail o el implícito en sale_dollars/sale_bottles) para calcular 
el margen de beneficio bruto por artículo, categoría y vendedor. 
• Optimización de Precios: Analizar la relación entre el precio minorista y el 
volumen de venta para sugerir ajustes de precios. 

# **1. Análisis de Rendimiento y Tendencias de Ventas**

## **Tendencias Temporales**
Tendencias Temporales (Ventas por Año)

In [0]:
%sql
SELECT
  d.year,
  SUM(f.sale_dollars) AS total_sales
FROM
  iowa_sales.sales_gold.fact_sales AS f
  JOIN iowa_sales.sales_gold.dim_date AS d ON f.date_key = d.date_key
GROUP BY
  d.year
ORDER BY
  d.year;

Tendencias Temporales (Ventas por Mes)

In [0]:
%sql
SELECT
  d.year,
  d.month,
  d.month_name,
  SUM(f.sale_dollars) AS total_sales
FROM
  iowa_sales.sales_gold.fact_sales AS f
  JOIN iowa_sales.sales_gold.dim_date AS d ON f.date_key = d.date_key
GROUP BY
  d.year,
  d.month,
  d.month_name
ORDER BY
  d.year,
  d.month;

Tendencias Temporales (Ventas por Día)

In [0]:
%sql
SELECT
  d.full_date,
  d.day_name,
  SUM(f.sale_dollars) AS total_sales
FROM
  iowa_sales.sales_gold.fact_sales AS f
  JOIN iowa_sales.sales_gold.dim_date AS d ON f.date_key = d.date_key
GROUP BY
  d.full_date,
  d.day_name
ORDER BY
  d.full_date;

In [0]:

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter

# --- Configuración de Estilo y Formato ---

# Establecer un estilo visual agradable para los gráficos
sns.set_style("whitegrid")

# Función para formatear el eje Y en Millones (ej. "200M")
def millions_formatter(x, pos):
    'Devuelve un string formateado en Millones'
    return f'{x/1_000_000:.0f}M'

formatter = FuncFormatter(millions_formatter)

# -------------------------------------------------------------------
# --- GRÁFICO 1: Tendencia de Ventas Anuales (Alto Nivel) ---
# -------------------------------------------------------------------
print("Generando Gráfico 1: Ventas Anuales...")

# 1. Definir la consulta SQL para ventas anuales
sql_anual = """
SELECT
  d.year,
  SUM(f.sale_dollars) AS total_sales
FROM
  iowa_sales.sales_gold.fact_sales AS f
  JOIN iowa_sales.sales_gold.dim_date AS d ON f.date_key = d.date_key
GROUP BY
  d.year
ORDER BY
  d.year
"""

# 2. Ejecutar la consulta y convertir a Pandas
df_anual = spark.sql(sql_anual).toPandas()

# 3. Crear el gráfico
fig_anual, ax_anual = plt.subplots(figsize=(12, 6))

# 4. Usar Seaborn para un gráfico de barras
sns.barplot(
    data=df_anual,
    x='year',
    y='total_sales',
    ax=ax_anual,
    palette='Blues_d'
)

# 5. Personalizar el gráfico
ax_anual.set_title('Tendencia de Ventas Anuales', fontsize=16, weight='bold')
ax_anual.set_xlabel('Año', fontsize=12)
ax_anual.set_ylabel('Ventas Totales (en Millones)', fontsize=12)
ax_anual.yaxis.set_major_formatter(formatter) # Aplicar formato "M"

# 6. Mostrar el gráfico en Databricks
display(fig_anual)
plt.close(fig_anual) # Cerrar la figura para liberar memoria

# -------------------------------------------------------------------
# --- GRÁFICO 2: Tendencia de Ventas Mensuales (Detallado) ---
# -------------------------------------------------------------------
print("Generando Gráfico 2: Ventas Mensuales...")

# 1. Usar la consulta optimizada para series de tiempo (YYYY-MM)
sql_mensual = """
SELECT
  DATE_FORMAT(d.full_date, 'yyyy-MM') AS year_month,
  SUM(f.sale_dollars) AS total_sales
FROM
  iowa_sales.sales_gold.fact_sales AS f
  JOIN iowa_sales.sales_gold.dim_date AS d ON f.date_key = d.date_key
GROUP BY
  year_month
ORDER BY
  year_month
"""

# 2. Ejecutar y convertir a Pandas
df_mensual = spark.sql(sql_mensual).toPandas()

# 3. Crear el gráfico (más ancho para series de tiempo)
fig_mensual, ax_mensual = plt.subplots(figsize=(20, 7))

# 4. Usar Seaborn para un gráfico de líneas
sns.lineplot(
    data=df_mensual,
    x='year_month',
    y='total_sales',
    ax=ax_mensual,
    marker='o' # Añadir marcadores en los puntos de datos
)

# 5. Personalizar el gráfico
ax_mensual.set_title('Tendencia de Ventas Mensuales (yyyy-MM)', fontsize=16, weight='bold')
ax_anual.set_xlabel('Año-Mes', fontsize=12)
ax_mensual.set_ylabel('Ventas Totales (en Millones)', fontsize=12)
ax_mensual.yaxis.set_major_formatter(formatter)

# 5b. Optimizar el eje X (es muy denso)
# Mostrar solo una etiqueta por cada 12 meses (1 año)
ax_mensual.set_xticks(df_mensual.index[::12])
ax_mensual.set_xticklabels(df_mensual['year_month'][::12], rotation=45, ha='right')

plt.tight_layout() # Ajustar para que las etiquetas no se corten

# 6. Mostrar el gráfico
display(fig_mensual)
plt.close(fig_mensual)

# -------------------------------------------------------------------
# --- GRÁFICO 3: Ventas por Día de la Semana (Patrones) ---
# -------------------------------------------------------------------
print("Generando Gráfico 3: Patrón de Ventas Semanales...")

# 1. Consulta para agrupar por día de la semana (ordenado)
sql_dia_semana = """
SELECT
  d.day_of_week, -- Para ordenar
  d.day_name,    -- Para etiquetar
  SUM(f.sale_dollars) AS total_sales
FROM
  iowa_sales.sales_gold.fact_sales AS f
  JOIN iowa_sales.sales_gold.dim_date AS d ON f.date_key = d.date_key
GROUP BY
  d.day_of_week,
  d.day_name
ORDER BY
  d.day_of_week -- Asegura que esté en orden (Lun, Mar, Mie...)
"""

# 2. Ejecutar y convertir a Pandas
df_dia_semana = spark.sql(sql_dia_semana).toPandas()

# 3. Crear el gráfico
fig_dia, ax_dia = plt.subplots(figsize=(12, 6))

# 4. Usar Seaborn para un gráfico de barras
sns.barplot(
    data=df_dia_semana,
    x='day_name',
    y='total_sales',
    ax=ax_dia,
    palette='Greens_d'
)

# 5. Personalizar el gráfico
ax_dia.set_title('Ventas Totales por Día de la Semana', fontsize=16, weight='bold')
ax_dia.set_xlabel('Día de la Semana', fontsize=12)
ax_dia.set_ylabel('Ventas Totales (en Millones)', fontsize=12)
ax_dia.yaxis.set_major_formatter(formatter)

# 6. Mostrar el gráfico
display(fig_dia)
plt.close(fig_dia)


## **Ranking de Puntos de Venta**
Top 20 Tiendas con Mayor Volumen de Ventas

In [0]:
%sql
SELECT
  s.store_name,
  s.city,
  SUM(f.sale_dollars) AS total_sales
FROM
  iowa_sales.sales_gold.fact_sales AS f
  JOIN iowa_sales.sales_gold.dim_store AS s ON f.store_key = s.store_key
GROUP BY
  s.store_name,
  s.city
ORDER BY
  total_sales DESC
LIMIT 20;

Top 20 Condados con Mayor Volumen de Ventas

In [0]:
%sql
SELECT
  s.county,
  SUM(f.sale_dollars) AS total_sales
FROM
  iowa_sales.sales_gold.fact_sales AS f
  JOIN iowa_sales.sales_gold.dim_store AS s ON f.store_key = s.store_key
WHERE
  s.county IS NOT NULL
  AND s.county <> 'UNKNOWN COUNTY'
GROUP BY
  s.county
ORDER BY
  total_sales DESC
LIMIT 20;

## **Comportamiento de Categorías**
Categorias con Mayor Volumen de Ventas y Numero de Botellas Total

In [0]:
%sql
SELECT
  p.category_name,
  SUM(f.sale_dollars) AS total_sales,
  SUM(f.bottles_sold) AS total_bottles
FROM
  iowa_sales.sales_gold.fact_sales AS f
  JOIN iowa_sales.sales_gold.dim_product AS p ON f.product_key = p.product_key
GROUP BY
  p.category_name
ORDER BY
  total_sales DESC
LIMIT 20;

Categorias con Mayor Crecimiento

La siguiente consulta está diseñada para mostrar el crecimiento año tras año de todas las categorias proporcionando un analisis mas detallado.

In [0]:
%sql
WITH CategorySalesByYear AS (
  -- Agrupar ventas totales por categoría y año
  SELECT
    p.category_name,
    d.year,
    SUM(f.sale_dollars) AS total_sales
  FROM
    iowa_sales.sales_gold.fact_sales AS f
    JOIN iowa_sales.sales_gold.dim_product AS p ON f.product_key = p.product_key
    JOIN iowa_sales.sales_gold.dim_date AS d ON f.date_key = d.date_key
  GROUP BY
    p.category_name,
    d.year
),
CategoryYoY AS (
  -- Usar LAG para obtener las ventas del año anterior
  SELECT
    category_name,
    year,
    total_sales,
    LAG(total_sales, 1, 0) OVER (
      PARTITION BY category_name
      ORDER BY year
    ) AS previous_year_sales
  FROM
    CategorySalesByYear
)
-- Calcular la tasa de crecimiento
SELECT
  category_name,
  year,
  total_sales,
  previous_year_sales,
  CASE
    WHEN previous_year_sales = 0 THEN NULL -- Evitar división por cero (categoría nueva)
    ELSE (total_sales - previous_year_sales) / previous_year_sales
  END AS yoy_growth_rate
FROM
  CategoryYoY
WHERE
  previous_year_sales >= 0 -- Mostrar solo categorías con datos del año anterior
ORDER BY
  category_name,year, yoy_growth_rate DESC,
  year DESC
LIMIT 50; 

La siguiente consulta está diseñada para mostrar el crecimiento porcentual total para cada categoría de producto, comparando las ventas de su primer año registrado contra las ventas de su último año registrado.

In [0]:
%sql
WITH CategorySalesByYear AS (
  -- Agrupar ventas totales por categoría y año (igual que antes)
  SELECT
    p.category_name,
    d.year,
    SUM(f.sale_dollars) AS total_sales
  FROM
    iowa_sales.sales_gold.fact_sales AS f
    JOIN iowa_sales.sales_gold.dim_product AS p ON f.product_key = p.product_key
    JOIN iowa_sales.sales_gold.dim_date AS d ON f.date_key = d.date_key
  GROUP BY
    p.category_name,
    d.year
),
RankedSales AS (
  -- Asignar un rango a cada año para cada categoría, en orden ascendente y descendente
  SELECT
    category_name,
    year,
    total_sales,
    -- El primer año va a tener rn_asc = 1
    ROW_NUMBER() OVER (PARTITION BY category_name ORDER BY year ASC) as rn_asc,
    -- El último año va a tener rn_desc = 1
    ROW_NUMBER() OVER (PARTITION BY category_name ORDER BY year DESC) as rn_desc
  FROM
    CategorySalesByYear
),
FirstLastSales AS (
  -- Filtrar para obtener solo las filas que son el primer o el último año
  SELECT
    category_name,
    year,
    total_sales,
    rn_asc,
    rn_desc
  FROM
    RankedSales
  WHERE
    rn_asc = 1 OR rn_desc = 1
)
-- Agrupar por categoría, pivotando los valores de primer/último año para calcular
SELECT
  category_name,
  
  -- Usar agregación condicional para obtener el primer año y sus ventas
  MIN(year) AS first_year,
  MAX(CASE WHEN rn_asc = 1 THEN total_sales END) AS first_year_sales,
  
  -- Usar agregación condicional para obtener el último año y sus ventas
  MAX(year) AS last_year,
  MAX(CASE WHEN rn_desc = 1 THEN total_sales END) AS last_year_sales,
  
  -- Calcular la tasa de crecimiento total
  CASE
    -- Si el primer y último año son el mismo (solo 1 año de datos), el crecimiento es 0
    WHEN MIN(year) = MAX(year) THEN 0
    -- Evitar división por cero si las ventas del primer año fueron 0
    WHEN MAX(CASE WHEN rn_asc = 1 THEN total_sales END) = 0 THEN NULL 
    ELSE 
      (MAX(CASE WHEN rn_desc = 1 THEN total_sales END) - MAX(CASE WHEN rn_asc = 1 THEN total_sales END)) 
      / MAX(CASE WHEN rn_asc = 1 THEN total_sales END)
  END AS total_growth_rate
FROM
  FirstLastSales
GROUP BY
  category_name
ORDER BY
  total_growth_rate DESC;

# **2. Análisis de Rentabilidad (Gross Margin)**

Cálculo de Margen Bruto (Por Artículo)

In [0]:
%sql
SELECT
  p.item_description,
  p.category_name,
  SUM(f.sale_dollars) AS total_revenue,
  SUM(f.state_bottle_cost * f.bottles_sold) AS total_cost,
  SUM(f.sale_dollars - (f.state_bottle_cost * f.bottles_sold)) AS gross_margin,
  CASE
    WHEN SUM(f.sale_dollars) = 0 THEN 0
    ELSE (SUM(f.sale_dollars - (f.state_bottle_cost * f.bottles_sold)) / SUM(f.sale_dollars))
  END AS margin_rate
FROM
  iowa_sales.sales_gold.fact_sales AS f
  JOIN iowa_sales.sales_gold.dim_product AS p ON f.product_key = p.product_key
GROUP BY
  p.item_description,
  p.category_name
ORDER BY
  gross_margin DESC
LIMIT 50;

Cálculo de Margen Bruto (Por Categoría)

In [0]:
%sql
SELECT
  p.category_name,
  SUM(f.sale_dollars) AS total_revenue,
  SUM(f.state_bottle_cost * f.bottles_sold) AS total_cost,
  SUM(f.sale_dollars - (f.state_bottle_cost * f.bottles_sold)) AS gross_margin,
  CASE
    WHEN SUM(f.sale_dollars) = 0 THEN 0
    ELSE (SUM(f.sale_dollars - (f.state_bottle_cost * f.bottles_sold)) / SUM(f.sale_dollars))
  END AS margin_rate
FROM
  iowa_sales.sales_gold.fact_sales AS f
  JOIN iowa_sales.sales_gold.dim_product AS p ON f.product_key = p.product_key
GROUP BY
  p.category_name
ORDER BY
  margin_rate DESC;

Cálculo de Margen Bruto (Por Vendedor)

In [0]:
%sql
SELECT
  v.vendor_name,
  SUM(f.sale_dollars) AS total_revenue,
  SUM(f.state_bottle_cost * f.bottles_sold) AS total_cost,
  SUM(f.sale_dollars - (f.state_bottle_cost * f.bottles_sold)) AS gross_margin,
  CASE
    WHEN SUM(f.sale_dollars) = 0 THEN 0
    ELSE (SUM(f.sale_dollars - (f.state_bottle_cost * f.bottles_sold)) / SUM(f.sale_dollars))
  END AS margin_rate
FROM
  iowa_sales.sales_gold.fact_sales AS f
  JOIN iowa_sales.sales_gold.dim_vendor AS v ON f.vendor_key = v.vendor_key
GROUP BY
  v.vendor_name
ORDER BY
  gross_margin DESC 
LIMIT 50;

## **Análisis de Optimización de Precios (Volumen por Punto de Precio)**
Este query analiza la relación entre el precio y el volumen de ventas para todos los productos que han tenido más de un precio. Agrupa el total histórico de botellas vendidas e ingresos para cada punto de precio de un producto, permitiendo ver qué precio generó la mayor demanda o los mayores ingresos en general.

In [0]:
%sql
-- CTE 1: Agrupar las ventas totales por producto y punto de precio (sin año)
WITH SalesByPricePoint AS (
  SELECT
    p.item_description,
    f.state_bottle_retail AS retail_price_point,
    SUM(f.bottles_sold) AS total_bottles_sold,
    SUM(f.sale_dollars) AS total_revenue_at_price
  FROM
    iowa_sales.sales_gold.fact_sales AS f
    JOIN iowa_sales.sales_gold.dim_product AS p ON f.product_key = p.product_key
  GROUP BY
    p.item_description,
    f.state_bottle_retail
),
-- CTE 2: Calcular cuántos precios distintos tiene cada producto
ProductPriceCounts AS (
  SELECT
    p.item_description,
    COUNT(DISTINCT f.state_bottle_retail) AS distinct_price_count
  FROM
    iowa_sales.sales_gold.fact_sales AS f
    JOIN iowa_sales.sales_gold.dim_product AS p ON f.product_key = p.product_key
  GROUP BY
    p.item_description
)
-- Selección final: Unir las ventas agregadas con los conteos de precios
SELECT
  s.item_description,
  s.retail_price_point,
  s.total_bottles_sold,
  s.total_revenue_at_price
FROM
  SalesByPricePoint AS s
  JOIN ProductPriceCounts AS c ON s.item_description = c.item_description
WHERE
  -- Filtrar para incluir solo productos con más de 1 precio distinto
  c.distinct_price_count > 1
ORDER BY
  s.item_description,
  s.retail_price_point;

In [0]:
%sql
select * from iowa_sales.sales_gold.fact_sales order by sale_id LIMIT 10 

In [0]:
%sql
select count(*) from iowa_sales.sales_gold.fact_sales;

In [0]:
%sql
select * from iowa_sales.sales_gold.dim_product LIMIT 10 

In [0]:
%sql
select count(*) from iowa_sales.sales_gold.dim_product;

In [0]:
%sql
select * from iowa_sales.sales_gold.dim_store LIMIT 10 

In [0]:
%sql
select count(*) from iowa_sales.sales_gold.dim_store;

In [0]:
%sql
select * from iowa_sales.sales_gold.dim_vendor LIMIT 10 

In [0]:
%sql
select count(*) from iowa_sales.sales_gold.dim_vendor;

In [0]:
%sql
SELECT
  f.sale_id,
  f.sale_invoice_line_no,
  f.date_key,
  d.full_date,
  d.year,
  d.month,
  d.day,
  f.product_key,
  p.item_description,
  p.category_name,
  f.store_key,
  s.store_key,
  s.store_name,
  s.city,
  s.zip_code,
  f.vendor_key,
  v.vendor_key,
  v.vendor_name,
  f.state_bottle_cost,
  f.state_bottle_retail,
  f.bottles_sold,
  f.sale_dollars,
  f.volume_sold_liters,
  f.volume_sold_gallons
FROM
  iowa_sales.sales_gold.fact_sales AS f
  JOIN iowa_sales.sales_gold.dim_product AS p ON f.product_key = p.product_key
  JOIN iowa_sales.sales_gold.dim_date AS d ON f.date_key = d.date_key
  JOIN iowa_sales.sales_gold.dim_store AS s ON f.store_key = s.store_key
  JOIN iowa_sales.sales_gold.dim_vendor AS v ON f.vendor_key = v.vendor_key
WHERE
  f.sale_invoice_line_no = '306831300004'


In [0]:
dbutils.fs.cp(
    "dbfs:/Volumes/dev/academy/data/iowa_sales_part_1.csv",
    "dbfs:/Volumes/iowa_sales/sales_bronze/process/iowa_sales_part_1.csv"
)

In [0]:
%sql
select saved_date from iowa_sales.sales_gold.fact_sales where sale_invoice_line_no = 'INV-14941300020'